In [1]:
import matplotlib.pyplot as plt
import datetime
import numpy as np
import time

In [2]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import site_archive_jasmin # Required to not get missing something error

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/ssde/j25a/mmh_storage/theme3/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25a/mmh_storage/theme3/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25a/mmh_storage/theme3/ra22_himawari', 'Rainfields3': '/gws/ssde/j25a/mmh_storage/theme3/rq0_rainfields_prcp_crate'}


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
selected_date = datetime.datetime(2021,6,9,2,0)
himawari = petdata.archive.Himawari('surface_global_irradiance')
rf3proj = petdata.transforms.projection.Rainfields3ProjAus()
radar_projector = petdata.transforms.projection.XYtoLonLatRectilinear(rf3proj)
satpipe = petpipe.Pipeline(
    himawari
)


In [6]:
fullsat = petpipe.Pipeline(
    satpipe,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20200101T00', '20210101T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

In [7]:
fullsat["20230117"]

/home/users/train097/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:326: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  full_ds = xr.open_mfdataset(
/home/users/train097/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:209: IndexWarning: Unable to subset on time dimension, returning all timesteps for validation. For request: 2023-01-17T00:00 -> 2023-01-18T00:00 @ 0 days 00:10:00
  warnings.warn(


array([[[[0.34095   , 0.30958333, 0.300125  , ..., 0.80073333,
          0.80134167, 0.800525  ],
         [0.32956667, 0.30050833, 0.32358333, ..., 0.79945   ,
          0.79945   , 0.79996667],
         [0.46000833, 0.35626667, 0.32355   , ..., 0.800425  ,
          0.79975833, 0.79975833],
         ...,
         [0.6813    , 0.6815    , 0.6817    , ..., 0.36261667,
          0.3529    , 0.25696667],
         [0.68135833, 0.68144167, 0.681675  , ..., 0.36566667,
          0.37875   , 0.365825  ],
         [0.68135833, 0.68144167, 0.681675  , ..., 0.26684167,
          0.36304167, 0.32965833]]],


       [[[0.37539167, 0.36935833, 0.364675  , ..., 0.817675  ,
          0.818275  , 0.81744167],
         [0.36734167, 0.3569    , 0.39174167, ..., 0.81636667,
          0.81636667, 0.81686667],
         [0.32715   , 0.35226667, 0.342825  , ..., 0.81735833,
          0.81666667, 0.81666667],
         ...,
         [0.70610833, 0.7063    , 0.70649167, ..., 0.33855   ,
          0.34040833, 0

In [15]:
# Reminder, the image size is latitude: 1726, longitude: 2214

class AutoEncoder(nn.Module):
    def __init__(self, 
                 input_height = 501,
                 input_width = 601,
                 kernel_size = 4,
                 stride=2,
                 input_channel_count = 2,
                 output_channel_count = 2,
                 latent_dim=300):
        super(AutoEncoder, self).__init__()

        self.input_width = input_width
        self.input_height = input_height
        self.input_channel_count = input_channel_count
        self.output_channel_count = output_channel_count

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=self.input_channel_count, out_channels=16, kernel_size=kernel_size, stride = stride, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride =2, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=7),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=self.output_channel_count, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):

        # Get latent representation
        latent = self.encoder(x)

        # Reconstruct input
        reconstructed = self.decoder(latent)

        return reconstructed

In [16]:
model = AutoEncoder(input_channel_count=1, output_channel_count=1).to(device)

In [17]:
# Loss function and optimizer
criterion = nn.L1Loss()
# criterion = nn.KLDivLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [18]:
import asyncio

async def get_next_batch(ipipe, sample_ix, max_samples, batch_size):
    batch = []
    while True:
        try:
            sample = next(ipipe)
        except StopIteration:
            break

        if torch.isnan(torch.tensor(sample)).any():
            continue

        if sample_ix > max_samples:
            break
        
        # Assume sample[0] is the input (e.g., image or tensor)
        batch.append(sample[0])
        sample_ix += 1

        if len(batch) == batch_size:
            break
        
    return np.stack(batch)

In [19]:
async def train(debug=True, num_epochs=1, max_samples=10, print_per=20, batch_size=16):
    """
    Main Training loop Function
    """
    print("a")
    
    sample_ix = 0
    for epoch in range(num_epochs):
        total_loss = 0
        epoch_samples = 0
        ipipe = iter(fullsat)  # Make an iterator to walk the time period
        print("b")


        next_batch_task = asyncio.create_task(get_next_batch(ipipe, sample_ix, max_samples, batch_size))
        print("c")
        start_time = time.time()
        while True:
            task_start_time = time.time()
            batch = await next_batch_task
            batch_load_time = time.time() - task_start_time
            print("d")


            next_batch_task = asyncio.create_task(get_next_batch(ipipe, sample_ix, max_samples, batch_size))
            
            sample_ix += batch.shape[0]
            epoch_samples += 1
            print(sample_ix)

            x = torch.from_numpy(batch).float().to(device)

            print("e")
    
            optimizer.zero_grad()
    
            # Forward pass
            y = model.forward(x)
            loss = criterion(y, x)
    
            # Backward pass and optimize        
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()

            batch = []

            if epoch_samples % print_per == 0:
                sample_train_run_time = time.time() - start_time
                print(f"[Epoch {epoch+1}] Sample {epoch_samples}, Batch Loss: {loss.item():.4f}, Load wait time: {batch_load_time:.2f}, Actual Train time: {sample_train_run_time - batch_load_time:.2f}")
                start_time = time.time()
    
        # Print epoch statistics
        avg_loss = total_loss / epoch_samples
        epoch_samples = 0  # Reset for next epoch
        print(f'Epoch [{epoch+1}/{epoch_samples}], Average Loss: {avg_loss:.4f}')

In [20]:
# Note %%time will not work with this use of await
await train(debug=False, num_epochs=1, max_samples=5000, print_per = 1, batch_size=1)

a
b
c
d
1
e
[Epoch 1] Sample 1, Batch Loss: 0.2510, Load wait time: 0.13, Actual Train time: 0.01
d
2
e
[Epoch 1] Sample 2, Batch Loss: 0.2674, Load wait time: 0.11, Actual Train time: 0.01
d
3
e
[Epoch 1] Sample 3, Batch Loss: 0.2729, Load wait time: 0.11, Actual Train time: 0.01
d
4
e
[Epoch 1] Sample 4, Batch Loss: 0.2659, Load wait time: 0.11, Actual Train time: 0.01
d
5
e
[Epoch 1] Sample 5, Batch Loss: 0.2375, Load wait time: 0.11, Actual Train time: 0.01
d
6
e
[Epoch 1] Sample 6, Batch Loss: 0.1969, Load wait time: 0.11, Actual Train time: 0.01
d
7
e
[Epoch 1] Sample 7, Batch Loss: 0.1782, Load wait time: 0.11, Actual Train time: 0.01
d
8
e
[Epoch 1] Sample 8, Batch Loss: 0.1880, Load wait time: 0.11, Actual Train time: 0.01
d
9
e
[Epoch 1] Sample 9, Batch Loss: 0.1721, Load wait time: 0.12, Actual Train time: 0.01
d
10
e
[Epoch 1] Sample 10, Batch Loss: 0.1468, Load wait time: 0.11, Actual Train time: 0.01
d
11
e
[Epoch 1] Sample 11, Batch Loss: 0.1312, Load wait time: 0.11, Ac

CancelledError: 

In [ ]:
fullsat_validate = petpipe.Pipeline(
    satpipe,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20210101T00', '20220101T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

In [ ]:
training_iterator = iter(fullsat) 
sample_numpy = next(training_iterator)

validate_iterator = iter(fullsat_validate)

def next_sample_gpu(pet_iterator):
    return torch.from_numpy(next(pet_iterator)).float().to(device)

sample_tensor_gpu = torch.from_numpy(sample_numpy).float().to(device)
sample_prediction_gpu = model.forward(sample_tensor_gpu)
sample_prediction_gpu = model.forward(next_sample_gpu(training_iterator))

def prediction_from_iterator(pet_iterator, model):
    sample_numpy = next(pet_iterator)
    sample_gpu = torch.from_numpy(sample_numpy).float().to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu = sample_prediction_gpu.to('cpu').detach().numpy()
    return prediction_cpu

prediction_from_iterator(validate_iterator, model)

In [ ]:
fig1 = plt.figure('sidebyside_satellite', figsize=(24,8))
for ix1 in range(3):
    sample_numpy = next(validate_iterator)
    sample_gpu = torch.from_numpy(sample_numpy).float().to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu = prediction_gpu.to('cpu').detach().numpy()
    ax1 = fig1.add_subplot(3,3,(ix1*3)+1)
    ax1.imshow(sample_numpy[0,0])
    ax1 = fig1.add_subplot(3,3,(ix1*3)+2)
    ax1.imshow(prediction_cpu[0,0])
    ax1 = fig1.add_subplot(3,3,(ix1*3)+3)
    ax1.imshow(prediction_cpu[0,0] - sample_numpy[0,0])  

In [ ]:
import torchmetrics.image
ssi_metric = torchmetrics.image.StructuralSimilarityIndexMeasure()
rmse_sw_metric = torchmetrics.image.RootMeanSquaredErrorUsingSlidingWindow()

ssi_values = []
rmse_values = []
for ix1 in range(10):
    sample_numpy = next(validate_iterator)
    sample_cpu = torch.from_numpy(sample_numpy).float()
    sample_gpu = sample_cpu.to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu_tensor = prediction_gpu.to('cpu').detach()
    
    ssi_values += [float(ssi_metric(sample_cpu,  prediction_cpu_tensor)) ]
    rmse_values += [float(rmse_sw_metric(sample_cpu,  prediction_cpu_tensor))]

def mean(list_):
    print(type(list_))
    return sum(list_)/len(list_)

print(f"Mean SSE {mean(ssi_values)}")
print(f"Mean RMSE {mean(rmse_values)}")